# Laboratoarele 4 și 5 — Modele de difuzie

**Lucrarea mea: DDPM pe Fashion-MNIST și experimente cu Stable Diffusion 1.5**

În notebookul acesta parcurg ambele laboratoare din cerință. În prima parte implementez un DDPM de la zero și îl antrenez pe Fashion-MNIST. În partea a doua folosesc un model Stable Diffusion preantrenat și măsor efectul unor setări de generare.

## Cuprins

- **Configurare** — biblioteci, seed, directoare
- **Laboratorul 4 — DDPM de la zero:** date, proces direct, U-Net, antrenare, generare DDPM/DDIM, comparația liniar-cosinus, bonus și întrebări
- **Laboratorul 5 — Stable Diffusion:** imagini de bază, instrumente de măsurare, experimente A–E, recomandare, img2img și întrebări
- **Concluziile mele** și salvarea figurilor/tabelelor

Celulele de cod sunt pregătite pentru rulare în Google Colab cu GPU T4 și acces la internet. Laboratorul 5 descarcă modelul și CLIP, iar rularea completă poate dura. Rezultatele și observațiile din raport trebuie să provină din rularea efectivă a celulelor.

# 0. Configurarea mediului

## 0.1 Biblioteci

În Colab, PyTorch, torchvision și Matplotlib sunt disponibile în mod obișnuit. Instalez pachetele folosite în partea Stable Diffusion. Fixez `transformers<5` pentru compatibilitate cu versiunea de `torchmetrics` folosită aici.

In [ ]:
!pip install -q "transformers<5" diffusers accelerate torchmetrics safetensors

## 0.2 Seed și directoare

Fixez seed-ul la 42 ca să pot reproduce comparațiile. Creez directoare pentru date, modele, grafice, tabele și imaginile generate. Dacă rulez notebookul în Colab, rezultatele se salvează în Google Drive.

In [ ]:
import os, sys, gc, json, math, time, random, shutil, functools, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from PIL import Image
from IPython.display import display

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FORTEAZA = False        # True = se reantrenează / se regenerează chiar dacă există fișiere salvate

# Directoare (în Colab se folosește Google Drive)
BAZA = "Lab4_5"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BAZA = "/content/drive/MyDrive/Lab4_5"
except Exception:
    print("Google Drive nu este disponibil; se folosește directorul local.")

DATA_DIR = os.path.join(BAZA, "Data_Lab4_5")
REZ_DIR = os.path.join(BAZA, "Rezultate_Lab4_5")
MODELE_DIR = os.path.join(REZ_DIR, "modele")
FIG_DIR = os.path.join(REZ_DIR, "figuri")
TAB_DIR = os.path.join(REZ_DIR, "tabele")
IMG5_DIR = os.path.join(REZ_DIR, "imagini_lab5")
for d in [DATA_DIR, MODELE_DIR, FIG_DIR, TAB_DIR, IMG5_DIR]:
    os.makedirs(d, exist_ok=True)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sincronizeaza():
    # GPU-ul lucrează asincron; înainte de citirea timpului se așteaptă terminarea calculelor
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def salveaza_fig(fig, nume):
    fig.savefig(os.path.join(FIG_DIR, nume + ".png"), dpi=150, bbox_inches="tight")


def salveaza_json(obiect, cale):
    with open(cale, "w", encoding="utf-8") as f:
        json.dump(obiect, f, ensure_ascii=False, indent=1)


def incarca_json(cale):
    with open(cale, "r", encoding="utf-8") as f:
        return json.load(f)


set_seed(SEED)
print("Dispozitiv:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "fără GPU (lent)")
print("Rezultate salvate în:", os.path.abspath(REZ_DIR))

# PARTEA I — Laboratorul 4: DDPM de la zero

În prima parte construiesc un model care învață să elimine zgomotul din imagini. Procesul direct adaugă zgomot gaussian în 1000 de pași. U-Net-ul învață să prezică zgomotul pentru o imagine și un pas date. Procesul invers folosește predicțiile pentru a porni de la zgomot și a obține imagini Fashion-MNIST.

## 1. Datele

Folosesc Fashion-MNIST: 60.000 de imagini pentru antrenare și 10.000 pentru test. Imaginile sunt grayscale, 28×28. Le măresc la 32×32 cu un chenar de 2 pixeli, le convertesc în tensori și normalizez pixelii în `[-1, 1]`. Dimensiunea 32 permite U-Net-ului să reducă rezoluția de trei ori: 32 → 16 → 8 → 4.

Încarc loturi de 128 de imagini și inspectez un lot înainte de antrenare.

In [ ]:
transform = transforms.Compose([
    transforms.Pad(2),                       # 28x28 -> 32x32
    transforms.ToTensor(),                   # valori în [0, 1]
    transforms.Normalize((0.5,), (0.5,)),    # valori în [-1, 1]
])
train_set = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=transform)
test_set = datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=transform)

BATCH = 128
loader = DataLoader(train_set, batch_size=BATCH, shuffle=True, num_workers=2,
                    generator=torch.Generator().manual_seed(SEED))

NUME_CLASE = ["Tricou", "Pantaloni", "Pulover", "Rochie", "Palton",
              "Sandală", "Cămașă", "Adidas", "Geantă", "Gheată"]


def grila_numpy(x, nrow=8):
    # Lipește un lot de imagini din [-1, 1] într-o grilă care poate fi afișată cu imshow
    x = (x.detach().cpu().float().clamp(-1, 1) + 1) / 2
    return make_grid(x, nrow=nrow, padding=2, pad_value=1.0).permute(1, 2, 0).numpy()


x_batch, y_batch = next(iter(loader))
print("Forma unui lot:", tuple(x_batch.shape), "| pixeli între", float(x_batch.min()), "și", float(x_batch.max()))
print("Loturi pe epocă:", len(loader))

fig, axs = plt.subplots(4, 4, figsize=(6.4, 7))
for i, ax in enumerate(axs.ravel()):
    ax.imshow(x_batch[i, 0], cmap="gray", vmin=-1, vmax=1)
    ax.set_title(NUME_CLASE[int(y_batch[i])], fontsize=9)
    ax.axis("off")
fig.suptitle("Fashion-MNIST: 16 imagini de antrenare (32×32)", y=0.94)
plt.tight_layout()
salveaza_fig(fig, "lab4_s1_date")
plt.show()

## 2. Procesul direct

Definesc două programe de zgomot (`beta`): unul liniar, de la 0,0001 la 0,02, și unul cosinus cu `s=0,008`. Din fiecare `beta_t` calculez `alpha_t = 1 - beta_t` și produsul cumulativ `alpha_bar_t`.

Pentru o imagine curată `x0`, zgomotul `epsilon` și pasul `t`, folosesc:

`x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * epsilon`

Formula îmi permite să obțin direct imaginea zgomotată la orice pas. În cod, indicele 0 reprezintă primul pas.

In [ ]:
T = 1000


def program_liniar(T=1000):
    return torch.linspace(0.0001, 0.02, T, dtype=torch.float64)


def program_cosinus(T=1000, s=0.008):
    t = torch.arange(T + 1, dtype=torch.float64)                      # t = 0, 1, ..., T
    f = torch.cos(((t / T + s) / (1 + s)) * math.pi / 2) ** 2
    alpha_bar = f / f[0]
    betas = 1 - alpha_bar[1:] / alpha_bar[:-1]
    return betas.clamp(max=0.999)                                     # limitat superior la 0,999


def construieste_program(betas):
    # Din beta se calculează alpha și produsul cumulat alpha_bar; tensorii se mută pe GPU
    betas = betas.double()
    alphas = 1 - betas
    alpha_bar = torch.cumprod(alphas, dim=0)
    valori = {"betas": betas, "alphas": alphas, "alpha_bar": alpha_bar,
              "sqrt_ab": alpha_bar.sqrt(), "sqrt_1m_ab": (1 - alpha_bar).sqrt()}
    return {k: v.float().to(DEVICE) for k, v in valori.items()}


PROG = {"liniar": construieste_program(program_liniar(T)),
        "cosinus": construieste_program(program_cosinus(T))}


def q_sample(x0, t, eps, prog):
    # Imaginea zgomotoasă la pasul t: x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * eps
    a = prog["sqrt_ab"][t][:, None, None, None]
    b = prog["sqrt_1m_ab"][t][:, None, None, None]
    return a * x0 + b * eps


for nume, p in PROG.items():
    print(f"{nume:8s}: beta între {p['betas'].min():.5f} și {p['betas'].max():.3f} | "
          f"alpha_bar la t=999: {p['alpha_bar'][-1]:.2e}")

### 2.1 Imaginea la niveluri diferite de zgomot

Aplic același zgomot aceleiași imagini pentru ambele programe, la pașii ceruți. Compar rândurile pentru a vedea când se pierde structura imaginii.

In [ ]:
x0_demo = train_set[0][0].unsqueeze(0).to(DEVICE)         # o imagine, forma (1, 1, 32, 32)
pasi_afisati = [0, 50, 100, 250, 500, 750, 999]
set_seed(SEED)
eps_demo = torch.randn_like(x0_demo)                      # același zgomot pentru ambele programe

fig, axs = plt.subplots(2, len(pasi_afisati), figsize=(2 * len(pasi_afisati), 4.8))
for r, nume in enumerate(["liniar", "cosinus"]):
    for c, t in enumerate(pasi_afisati):
        xt = q_sample(x0_demo, torch.tensor([t], device=DEVICE), eps_demo, PROG[nume])
        axs[r, c].imshow(xt[0, 0].clamp(-1, 1).cpu(), cmap="gray", vmin=-1, vmax=1)
        axs[r, c].set_xticks([]); axs[r, c].set_yticks([])
        if r == 0:
            axs[r, c].set_title(f"t = {t}")
    axs[r, 0].set_ylabel(f"program {nume}", fontsize=11)
fig.suptitle("Procesul direct: aceeași imagine, același zgomot, două programe", y=1.0)
plt.tight_layout()
salveaza_fig(fig, "lab4_s2_proces_direct")
plt.show()

### 2.2 Cât semnal rămâne

Trasez `sqrt(alpha_bar_t)`, coeficientul imaginii originale. Apoi calculez primul pas la care fiecare curbă coboară sub 0,5. Folosesc această valoare ca reper pentru a compara ritmul celor două programe.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.3))
for nume, cul in [("liniar", "tab:blue"), ("cosinus", "tab:orange")]:
    ax.plot(range(T), PROG[nume]["sqrt_ab"].cpu(), label=f"program {nume}", color=cul, lw=2)
ax.axhline(0.5, color="gray", ls="--", lw=1, label="prag 0,5")
ax.set_xlabel("pasul t")
ax.set_ylabel(r"$\sqrt{\bar{\alpha}_t}$")
ax.set_title(r"Semnalul rămas din imaginea originală: $\sqrt{\bar{\alpha}_t}$")
ax.grid(alpha=0.3)
ax.legend()
salveaza_fig(fig, "lab4_s2_sqrt_alpha_bar")
plt.show()

for nume in PROG:
    pas = int((PROG[nume]["sqrt_ab"] < 0.5).nonzero()[0].item())     # primul t cu sqrt(alpha_bar) < 0,5
    print(f"Programul {nume:8s}: sqrt(alpha_bar) scade sub 0,5 la t = {pas}")

## 3. Rețeaua U-Net

U-Net-ul meu are o ramură descendentă care reduce rezoluția și extrage trăsături, apoi o ramură ascendentă care reface rezoluția. Blocurile reziduale ajută la propagarea informației, iar conexiunile skip transferă detaliile encoderului către decoder. Reprezentarea sinusoidală a pasului `t` este adăugată în blocurile reziduale, fiindcă nivelul de zgomot depinde de pas.

In [ ]:
def time_embedding(t, dim=128):
    # t: tensor (B,) cu pașii; întoarce (B, dim)  -> încorporarea sinusoidală a pasului t
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)


class ResBlock(nn.Module):
    def __init__(self, c_in, c_out, t_dim=128):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, c_in)
        self.conv1 = nn.Conv2d(c_in, c_out, 3, padding=1)
        self.t_proj = nn.Linear(t_dim, c_out)                     # <-- pasul t intră aici
        self.norm2 = nn.GroupNorm(8, c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1)
        self.skip = nn.Conv2d(c_in, c_out, 1) if c_in != c_out else nn.Identity()

    def forward(self, x, temb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.t_proj(temb)[:, :, None, None]               # <-- pasul t se adaugă la trăsături
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)                                   # <-- conexiune reziduală


class UNet(nn.Module):
    def __init__(self, ch=(32, 64, 128), t_dim=128):
        super().__init__()
        self.t_mlp = nn.Sequential(nn.Linear(t_dim, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim))
        self.inp = nn.Conv2d(1, ch[0], 3, padding=1)
        self.down1 = ResBlock(ch[0], ch[0])
        self.down2 = ResBlock(ch[0], ch[1])
        self.down3 = ResBlock(ch[1], ch[2])
        self.mid = ResBlock(ch[2], ch[2])
        self.up3 = ResBlock(2 * ch[2], ch[1])
        self.up2 = ResBlock(2 * ch[1], ch[0])
        self.up1 = ResBlock(2 * ch[0], ch[0])
        self.out = nn.Conv2d(ch[0], 1, 1)

    def forward(self, x, t):
        temb = self.t_mlp(time_embedding(t))                      # <-- pasul t: încorporare sinusoidală + MLP
        x = self.inp(x)
        s1 = self.down1(x, temb)                                  # 32×32
        s2 = self.down2(F.avg_pool2d(s1, 2), temb)                # 16×16
        s3 = self.down3(F.avg_pool2d(s2, 2), temb)                # 8×8
        h = self.mid(F.avg_pool2d(s3, 2), temb)                   # 4×4
        up = lambda z: F.interpolate(z, scale_factor=2)
        h = self.up3(torch.cat([up(h), s3], dim=1), temb)         # <-- conexiune skip (concatenare cu s3)
        h = self.up2(torch.cat([up(h), s2], dim=1), temb)         # <-- conexiune skip (concatenare cu s2)
        h = self.up1(torch.cat([up(h), s1], dim=1), temb)         # <-- conexiune skip (concatenare cu s1)
        return self.out(h)                                        # zgomotul prezis

### 3.1 Verificarea ieșirii

Trimit un lot artificial prin U-Net. Verific dacă ieșirea are forma `(8, 1, 32, 32)`, aceeași formă ca intrarea, și număr parametrii antrenabili.

In [ ]:
def nr_parametri(model):
    return sum(p.numel() for p in model.parameters())


model_test = UNet().to(DEVICE)
x_test = torch.randn(8, 1, 32, 32, device=DEVICE)
t_test = torch.randint(0, 1000, (8,), device=DEVICE)
print("Forma intrării:", tuple(x_test.shape), "| forma ieșirii:", tuple(model_test(x_test, t_test).shape))
print(f"Număr de parametri: {nr_parametri(model_test):,}")
del model_test

În implementare, urmăresc explicit trei componente:

| Componentă | Unde o identific în cod |
|---|---|
| Pasul `t` | `time_embedding` → `t_mlp` → `t_proj` din blocul rezidual |
| Conexiuni skip | caracteristicile `s1`, `s2`, `s3`, concatenate în decoder |
| Conexiuni reziduale | adunarea `h + self.skip(x)` în `ResBlock` |

## 4. Antrenarea DDPM cu programul liniar

La fiecare lot aleg câte un pas aleator pentru fiecare imagine, generez zgomot gaussian și construiesc `x_t`. U-Net-ul prezice zgomotul adăugat. Calculez eroarea pătratică medie dintre zgomotul real și cel prezis, apoi actualizez greutățile cu Adam.

Setările mele sunt rata de învățare `0,0002`, lot de 128, seed 42 și 15 epoci. Păstrez pierderea medie pe epocă și salvez modelul antrenat.

In [ ]:
EPOCI = 15
LR = 2e-4


def antreneaza(nume, prog, ch=(32, 64, 128)):
    # Antrenează un U-Net cu programul de zgomot `prog` sau îl încarcă dacă a fost deja salvat.
    # Întoarce modelul, pierderea medie pe epocă și timpul de antrenare (s).
    cale = os.path.join(MODELE_DIR, f"{nume}.pt")
    model = UNet(ch=ch).to(DEVICE)
    if os.path.exists(cale) and not FORTEAZA:
        st = torch.load(cale, map_location=DEVICE)
        model.load_state_dict(st["model"])
        print(f"[{nume}] model încărcat din {cale}")
        return model.eval(), st["pierderi"], st["timp"]

    set_seed(SEED)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    pierderi = []
    sincronizeaza()
    t0 = time.perf_counter()
    for ep in range(EPOCI):
        model.train()
        suma, n = 0.0, 0
        for x0, _ in loader:
            x0 = x0.to(DEVICE)
            t = torch.randint(0, T, (x0.size(0),), device=DEVICE)   # pas aleator pentru fiecare imagine
            eps = torch.randn_like(x0)                               # zgomotul real
            xt = q_sample(x0, t, eps, prog)                          # imaginea zgomotoasă
            loss = F.mse_loss(model(xt, t), eps)                     # rețeaua prezice zgomotul
            opt.zero_grad()
            loss.backward()
            opt.step()
            suma += loss.item() * x0.size(0)
            n += x0.size(0)
        pierderi.append(suma / n)
        print(f"[{nume}] epoca {ep + 1:2d}/{EPOCI} | pierdere medie {pierderi[-1]:.5f}")
    sincronizeaza()
    timp = time.perf_counter() - t0
    torch.save({"model": model.state_dict(), "pierderi": pierderi, "timp": timp, "ch": list(ch)}, cale)
    print(f"[{nume}] antrenare terminată în {timp / 60:.1f} min; model salvat în {cale}")
    return model.eval(), pierderi, timp


@torch.no_grad()
def pierdere_pe_niveluri(model, prog, niveluri=tuple(range(0, 1000, 50)) + (999,), n=512):
    # Pierderea MSE pe imagini de test, separat pentru fiecare nivel de zgomot t
    model.eval()
    x0 = torch.stack([test_set[i][0] for i in range(n)]).to(DEVICE)
    g = torch.Generator(device=DEVICE).manual_seed(0)
    eps = torch.randn(x0.shape, generator=g, device=DEVICE)
    rez = {}
    for t in niveluri:
        tt = torch.full((n,), t, device=DEVICE, dtype=torch.long)
        rez[t] = F.mse_loss(model(q_sample(x0, tt, eps, prog), tt), eps).item()
    return rez


model_lin, pierderi_lin, timp_lin = antreneaza("ddpm_liniar", PROG["liniar"])

### 4.1 Evoluția pierderii

Graficul arată pierderea de antrenare pe epoci. După rulare, notez dacă pierderea scade și dacă se stabilizează. Celulele de evaluare și graficare de mai jos îmi permit să verific și comportamentul pe niveluri diferite de zgomot.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11.5, 4.2))
ep = range(1, len(pierderi_lin) + 1)
axs[0].plot(ep, pierderi_lin, marker="o", color="tab:blue")
axs[0].set_xlabel("epoca")
axs[0].set_ylabel("pierdere medie (MSE)")
axs[0].set_title("Pierderea pe epocă (program liniar)")
axs[0].grid(alpha=0.3)

niv_lin = pierdere_pe_niveluri(model_lin, PROG["liniar"])
axs[1].plot(list(niv_lin), list(niv_lin.values()), marker="o", ms=4, color="tab:blue")
axs[1].set_yscale("log")
axs[1].set_xlabel("pasul t (nivelul de zgomot)")
axs[1].set_ylabel("MSE (scară logaritmică)")
axs[1].set_title("Pierderea pe niveluri de zgomot (imagini de test)")
axs[1].grid(alpha=0.3, which="both")
plt.tight_layout()
salveaza_fig(fig, "lab4_s4_pierdere_liniar")
plt.show()

p = pierderi_lin
print(f"Pierderea scade de la {p[0]:.4f} (epoca 1) la {p[-1]:.4f} (epoca {len(p)}), adică cu {100 * (1 - p[-1] / p[0]):.0f}%.")
print(f"Scădere în ultimele 3 epoci: {100 * (p[-4] - p[-1]) / p[-4]:.1f}%")
print(f"Timp total de antrenare: {timp_lin / 60:.1f} min ({timp_lin / len(p):.0f} s pe epocă)")
print("Cea mai mare eroare apare la t =", max(niv_lin, key=niv_lin.get))

## 5. Generarea DDPM

Implementez procesul invers DDPM pornind de la `x_T` extras din zgomot gaussian. Parcurg pașii de la 999 la 0; la fiecare pas, U-Net-ul prezice zgomotul și actualizez imaginea. Adaug zgomot aleator la pașii intermediari, dar nu la ultimul. Generez un lot de 64 de imagini și păstrez stări intermediare pentru a urmări transformarea.

In [ ]:
def zgomot_initial(n=64, seed=SEED):
    # Același zgomot de pornire la fiecare apel cu același seed
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    return torch.randn(n, 1, 32, 32, generator=g, device=DEVICE)


def masoara(functie, *args, **kwargs):
    # Rulează o funcție și întoarce (rezultat, timp în secunde), cu sincronizare GPU
    sincronizeaza()
    t0 = time.perf_counter()
    rezultat = functie(*args, **kwargs)
    sincronizeaza()
    return rezultat, time.perf_counter() - t0


@torch.no_grad()
def esantioneaza_ddpm(model, prog, x_T, seed=SEED + 1):
    model.eval()
    g = torch.Generator(device=DEVICE).manual_seed(seed)           # zgomotul z de la fiecare pas
    x = x_T.clone()
    stari = [("zgomot", x[0].cpu())]
    salvat = [800, 600, 400, 200, 100, 50, 0]
    for t in range(T - 1, -1, -1):
        tt = torch.full((x.size(0),), t, device=DEVICE, dtype=torch.long)
        eps = model(x, tt)                                          # zgomotul prezis
        beta, alfa, ab = prog["betas"][t], prog["alphas"][t], prog["alpha_bar"][t]
        medie = (x - beta / torch.sqrt(1 - ab) * eps) / torch.sqrt(alfa)
        if t > 0:
            x = medie + torch.sqrt(beta) * torch.randn(x.shape, generator=g, device=DEVICE)
        else:
            x = medie                                               # la t = 0 nu se adaugă zgomot
        if t in salvat:
            stari.append((f"după t={t}", x[0].cpu()))
    return x, stari


x_T = zgomot_initial(64)
esantioneaza_ddpm(model_lin, PROG["liniar"], x_T[:2])               # încălzire GPU, rezultatul se ignoră
(x_ddpm, stari_ddpm), timp_ddpm = masoara(esantioneaza_ddpm, model_lin, PROG["liniar"], x_T)
print(f"Generare DDPM (1000 pași, 64 de imagini): {timp_ddpm:.1f} s")

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(grila_numpy(x_ddpm))
ax.axis("off")
ax.set_title(f"64 de imagini generate cu DDPM (program liniar, 1000 pași, {timp_ddpm:.1f} s)")
salveaza_fig(fig, "lab4_s5_ddpm_grila")
plt.show()

fig, axs = plt.subplots(1, len(stari_ddpm), figsize=(2 * len(stari_ddpm), 2.6))
for ax, (eticheta, img) in zip(axs, stari_ddpm):
    ax.imshow(img[0].clamp(-1, 1), cmap="gray", vmin=-1, vmax=1)
    ax.set_title(eticheta, fontsize=9)
    ax.axis("off")
fig.suptitle("Stările intermediare ale unei imagini: de la zgomot la rezultat", y=1.05)
salveaza_fig(fig, "lab4_s5_stari_intermediare")
plt.show()

### 5.1 Timpul de calcul

Măsor timpul unui pas de antrenare și timpul necesar generării DDPM complete. Folosesc aceeași mașină și sincronizez GPU-ul înainte de citirea timpului, ca măsurarea să includă calculele terminate.

In [ ]:
def timp_pas_antrenare(model, prog, repetari=20):
    m = copy.deepcopy(model).train()
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    x0 = x_batch.to(DEVICE)

    def pas():
        t = torch.randint(0, T, (x0.size(0),), device=DEVICE)
        eps = torch.randn_like(x0)
        loss = F.mse_loss(m(q_sample(x0, t, eps, prog), t), eps)
        opt.zero_grad()
        loss.backward()
        opt.step()

    pas()                                   # încălzire
    sincronizeaza()
    t0 = time.perf_counter()
    for _ in range(repetari):
        pas()
    sincronizeaza()
    return (time.perf_counter() - t0) / repetari


t_pas = timp_pas_antrenare(model_lin, PROG["liniar"])
print(f"Un pas de antrenare (lot de {BATCH}): {t_pas * 1000:.1f} ms")
print(f"Generare DDPM completă (64 de imagini): {timp_ddpm:.1f} s")
print(f"Generarea durează de aproximativ {timp_ddpm / t_pas:.0f} ori mai mult decât un pas de antrenare.")

## 6. Generare mai rapidă cu DDIM

DDIM reutilizează U-Net-ul antrenat, dar sare peste o parte dintre pașii celor 1000. Folosesc același zgomot inițial pentru toate configurațiile, apoi compar 10, 20, 50, 200 și 1000 de pași.

La pași rari, estimarea imaginii curate `x0` poate depăși intervalul pixelilor `[-1, 1]`. Funcția include opțiunea de limitare a acestei estimări și recalcularea zgomotului. Compar efectul în celula dedicată.

In [ ]:
@torch.no_grad()
def esantioneaza_ddim(model, prog, x_T, n_pasi, limiteaza=True):
    model.eval()
    pasi = torch.linspace(T - 1, 0, n_pasi, dtype=torch.float64).round().long().tolist()
    x = x_T.clone()
    for i, t in enumerate(pasi):
        tt = torch.full((x.size(0),), t, device=DEVICE, dtype=torch.long)
        eps = model(x, tt)                                          # zgomotul prezis
        ab = prog["alpha_bar"][t]
        x0 = (x - torch.sqrt(1 - ab) * eps) / torch.sqrt(ab)        # estimarea imaginii curate
        if limiteaza:
            x0 = x0.clamp(-1, 1)                                    # pixelii reali sunt în [-1, 1]
            eps = (x - torch.sqrt(ab) * x0) / torch.sqrt(1 - ab)    # zgomot coerent cu x0 limitat
        if i == len(pasi) - 1:
            x = x0                                                  # ultimul pas: rezultatul este x0
        else:
            ab_urm = prog["alpha_bar"][pasi[i + 1]]
            x = torch.sqrt(ab_urm) * x0 + torch.sqrt(1 - ab_urm) * eps
    return x


def genereaza_ddim(model, prog, n, limiteaza=True):
    # 64 de imagini DDIM din același zgomot x_T; întoarce (imagini pe CPU, timp în secunde)
    x, dt = masoara(esantioneaza_ddim, model, prog, x_T, n, limiteaza)
    return x.clamp(-1, 1).cpu(), dt


def stat_pixeli(x):
    # media și abaterea standard a pixelilor, în [0, 1]
    im = torch.nan_to_num((x.detach().cpu().float().clamp(-1, 1) + 1) / 2)
    return float(im.mean()), float(im.std())


REAL_M, REAL_S = stat_pixeli(torch.stack([test_set[i][0] for i in range(512)]))

esantioneaza_ddim(model_lin, PROG["liniar"], x_T[:2], 5)            # încălzire GPU
PASI_DDIM = [1000, 200, 50, 20, 10]
rez_ddim = {}                                                       # n -> (imagini, timp)
for n in PASI_DDIM:
    rez_ddim[n] = genereaza_ddim(model_lin, PROG["liniar"], n)
    print(f"DDIM, {n:4d} pași, 64 de imagini: {rez_ddim[n][1]:6.2f} s")

### 6.1 Grilele DDIM

Compar vizual grilele de la 10 până la 1000 de pași. Pentru că zgomotul inițial este comun, aceeași poziție din grile arată efectul numărului de pași asupra aceluiași eșantion.

In [ ]:
panouri = [("DDPM, 1000 pași", x_ddpm.clamp(-1, 1).cpu(), timp_ddpm)] + \
          [(f"DDIM, {n} pași", rez_ddim[n][0], rez_ddim[n][1]) for n in PASI_DDIM]
fig, axs = plt.subplots(2, 3, figsize=(15, 10.5))
for ax, (titlu, x, dt) in zip(axs.ravel(), panouri):
    ax.imshow(grila_numpy(x))
    ax.set_title(f"{titlu} ({dt:.1f} s)", fontsize=12)
    ax.axis("off")
fig.suptitle("Program liniar: DDPM și DDIM, același zgomot de pornire", y=1.0, fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.97])
salveaza_fig(fig, "lab4_s6_ddim_grile")
plt.show()

### 6.2 Tabelul timpului și al diferenței

Tabelul compară DDPM și DDIM după timpul de generare și diferența pixelilor față de DDIM cu 1000 de pași. Include și media/abaterea standard a pixelilor, pe care le compar cu imaginile reale. Pentru DDPM, zgomotul aleator introdus la fiecare pas înseamnă că diferența pixel cu pixel nu este o comparație identică de eșantion.

In [ ]:
ref = rez_ddim[1000][0]
linii = []
for metoda, n, (x, dt) in [("DDPM", 1000, (x_ddpm.clamp(-1, 1).cpu(), timp_ddpm))] + [("DDIM", n, rez_ddim[n]) for n in PASI_DDIM]:
    m, s = stat_pixeli(x)
    linii.append({"Metodă": metoda, "Pași": n, "Timp pentru 64 de imagini (s)": round(dt, 2),
                  "MSE față de DDIM-1000": "—" if metoda == "DDPM" else round(float(((x - ref) ** 2).mean()), 4),
                  "Medie pixeli": round(m, 4), "Abatere std pixeli": round(s, 4)})
tabel_ddim = pd.DataFrame(linii)
display(tabel_ddim)
print(f"Imagini reale de test: medie pixeli = {REAL_M:.4f}, abatere std = {REAL_S:.4f}")
tabel_ddim.to_csv(os.path.join(TAB_DIR, "lab4_tabel_ddpm_ddim.csv"), index=False)

print("\nViteza față de DDPM cu 1000 de pași:")
for n in PASI_DDIM:
    print(f"  DDIM {n:4d} pași: {rez_ddim[n][1]:6.2f} s ({timp_ddpm / rez_ddim[n][1]:5.1f}× mai rapid)")

După rulare, completez evaluarea vizuală pe baza grilelor. Folosesc aceeași scară pentru toate metodele:

| Metodă | Pași | Calitate (1–5) | Observații din imaginile mele |
|---|---:|---:|---|
| DDPM | 1000 |  |  |
| DDIM | 1000 |  |  |
| DDIM | 200 |  |  |
| DDIM | 50 |  |  |
| DDIM | 20 |  |  |
| DDIM | 10 |  |  |

### 6.3 Efectul limitării lui `x0`

Cu același model și același zgomot de pornire, compar DDIM la 200 și 1000 de pași, cu limitarea lui `x0` activată și dezactivată. Compar grilele și statisticile pixelilor.

In [ ]:
rez_lim = {}
for limiteaza in (False, True):
    for n in (200, 1000):
        rez_lim[(limiteaza, n)] = rez_ddim[n][0] if limiteaza else genereaza_ddim(model_lin, PROG["liniar"], n, limiteaza=False)[0]

fig, axs = plt.subplots(2, 2, figsize=(11, 11))
linii = [{"Set": "imagini reale de test", "Medie pixeli": round(REAL_M, 4), "Abatere std pixeli": round(REAL_S, 4)}]
for r, limiteaza in enumerate((False, True)):
    for c, n in enumerate((200, 1000)):
        x = rez_lim[(limiteaza, n)]
        axs[r, c].imshow(grila_numpy(x))
        axs[r, c].set_title(f"{'cu' if limiteaza else 'fără'} limitare — DDIM, {n} pași", fontsize=12)
        axs[r, c].axis("off")
        m, s = stat_pixeli(x)
        linii.append({"Set": f"{'cu' if limiteaza else 'fără'} limitare, {n} pași",
                      "Medie pixeli": round(m, 4), "Abatere std pixeli": round(s, 4)})
fig.suptitle("DDIM fără și cu limitarea lui x₀ (program liniar)", y=1.0, fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.97])
salveaza_fig(fig, "lab4_s6_efect_limitare")
plt.show()

tabel_lim = pd.DataFrame(linii)
display(tabel_lim)
tabel_lim.to_csv(os.path.join(TAB_DIR, "lab4_efect_limitare_x0.csv"), index=False)

### 6.4 Timp, număr de pași și diferența față de DDIM-1000

Graficul reunește timpul și abaterea față de eșantionarea completă. Folosesc axa logaritmică pentru a putea citi intervalul larg de pași.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].plot(PASI_DDIM, [rez_ddim[n][1] for n in PASI_DDIM], marker="o")
axs[0].set_xscale("log"); axs[0].set_yscale("log")
axs[0].set_xlabel("număr de pași DDIM")
axs[0].set_ylabel("timp pentru 64 de imagini (s)")
axs[0].set_title("Timpul de generare")
axs[0].grid(alpha=0.3, which="both")

pasi_mse = PASI_DDIM[1:]
axs[1].plot(pasi_mse, [float(((rez_ddim[n][0] - ref) ** 2).mean()) for n in pasi_mse], marker="o", color="tab:red")
axs[1].set_xscale("log")
axs[1].set_xlabel("număr de pași DDIM")
axs[1].set_ylabel("MSE față de DDIM-1000")
axs[1].set_title("Diferența față de rezultatul cu 1000 de pași")
axs[1].grid(alpha=0.3, which="both")
plt.tight_layout()
salveaza_fig(fig, "lab4_s6_compromis_pasi")
plt.show()

## 7. Program liniar comparat cu programul cosinus

Antrenez un al doilea U-Net cu programul cosinus și aceleași setări ca la programul liniar. Generez imagini cu DDIM la 20 și 50 de pași, din același zgomot inițial. Pierderile brute nu sunt direct comparabile, fiindcă fiecare program expune modelul la distribuții diferite de niveluri de zgomot; îmi bazez comparația și pe grile și statisticile imaginilor.

In [ ]:
model_cos, pierderi_cos, timp_cos = antreneaza("ddpm_cosinus", PROG["cosinus"])
MODELE = {"liniar": model_lin, "cosinus": model_cos}

rez7 = {}                                                           # (program, pași) -> (imagini, timp)
for nume, model in MODELE.items():
    for n in (50, 20):
        rez7[(nume, n)] = rez_ddim[n] if nume == "liniar" else genereaza_ddim(model, PROG[nume], n)
        print(f"{nume:8s} | DDIM {n:3d} pași | {rez7[(nume, n)][1]:5.2f} s")

### 7.1 Grilele alăturate

Compar rezultatele liniare și cosinus pentru același număr de pași și același zgomot inițial.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(11.5, 11.5))
for r, n in enumerate((50, 20)):
    for c, nume in enumerate(("liniar", "cosinus")):
        axs[r, c].imshow(grila_numpy(rez7[(nume, n)][0]))
        axs[r, c].set_title(f"program {nume}, DDIM {n} pași", fontsize=12)
        axs[r, c].axis("off")
fig.suptitle("Liniar vs. cosinus: același zgomot de pornire, același număr de pași", y=1.0, fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.97])
salveaza_fig(fig, "lab4_s7_liniar_vs_cosinus")
plt.show()

Completez observațiile vizuale după ce rulez comparația:

| Program | Pași DDIM | Calitate (1–5) | Observații din imaginile mele |
|---|---:|---:|---|
| Liniar | 50 |  |  |
| Cosinus | 50 |  |  |
| Liniar | 20 |  |  |
| Cosinus | 20 |  |  |

### 7.2 Statisticile pixelilor

Compar media și abaterea standard a imaginilor reale și generate. Le folosesc ca indicatori suplimentari pentru luminozitate și contrast, nu ca înlocuitor al inspecției vizuale.

In [ ]:
linii = [{"Set": "imagini reale de test", "Medie pixeli": round(REAL_M, 4), "Abatere std pixeli": round(REAL_S, 4)}]
for n in (50, 20):
    for nume in ("liniar", "cosinus"):
        m, s = stat_pixeli(rez7[(nume, n)][0])
        linii.append({"Set": f"{nume}, DDIM {n} pași", "Medie pixeli": round(m, 4), "Abatere std pixeli": round(s, 4)})
tabel_pix = pd.DataFrame(linii)
display(tabel_pix)
tabel_pix.to_csv(os.path.join(TAB_DIR, "lab4_statistici_pixeli.csv"), index=False)

### 7.3 Curbele de pierdere

Pun curbele celor două antrenări pe același grafic și verific distribuția pierderii pe niveluri de zgomot. Interpretez pierderea în contextul fiecărui program, nu compar doar valorile finale.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.3))
axs[0].plot(range(1, EPOCI + 1), pierderi_lin, marker="o", label="program liniar", color="tab:blue")
axs[0].plot(range(1, EPOCI + 1), pierderi_cos, marker="o", label="program cosinus", color="tab:orange")
axs[0].set_xlabel("epoca")
axs[0].set_ylabel("pierdere medie (MSE)")
axs[0].set_title("Curbele de pierdere (valori necomparabile direct)")
axs[0].grid(alpha=0.3)
axs[0].legend()

niv_cos = pierdere_pe_niveluri(model_cos, PROG["cosinus"])
axs[1].plot(list(niv_lin), list(niv_lin.values()), marker="o", ms=4, label="program liniar", color="tab:blue")
axs[1].plot(list(niv_cos), list(niv_cos.values()), marker="o", ms=4, label="program cosinus", color="tab:orange")
axs[1].set_yscale("log")
axs[1].set_xlabel("pasul t (nivelul de zgomot)")
axs[1].set_ylabel("MSE (scară logaritmică)")
axs[1].set_title("Pierderea pe niveluri de zgomot (imagini de test)")
axs[1].grid(alpha=0.3, which="both")
axs[1].legend()
plt.tight_layout()
salveaza_fig(fig, "lab4_s7_curbe_pierdere")
plt.show()

print(f"Pierdere finală: liniar {pierderi_lin[-1]:.5f}, cosinus {pierderi_cos[-1]:.5f}")
print(f"Timp de antrenare: liniar {timp_lin / 60:.1f} min, cosinus {timp_cos / 60:.1f} min")
print(f"Cea mai mare eroare pe imagini de test: liniar la t = {max(niv_lin, key=niv_lin.get)}, cosinus la t = {max(niv_cos, key=niv_cos.get)}")

## 8. Bonus: U-Net mai mic

Antrenez o variantă cu mai puține canale (`16, 32, 64`) și o compar cu modelul de bază după numărul de parametri, timpul de antrenare, pierderea finală, timpul de generare și grila DDIM la 50 de pași.

In [ ]:
model_mic, pierderi_mic, timp_mic = antreneaza("ddpm_liniar_mic", PROG["liniar"], ch=(16, 32, 64))
x_mic, timp_gen_mic = genereaza_ddim(model_mic, PROG["liniar"], 50)
x_mare, timp_gen_mare = rez_ddim[50]

tabel_bonus = pd.DataFrame([
    {"Rețea": "UNet (32, 64, 128)", "Parametri": f"{nr_parametri(model_lin):,}", "Timp antrenare (min)": round(timp_lin / 60, 1),
     "Pierdere finală": round(pierderi_lin[-1], 5), "Timp DDIM-50 (s)": round(timp_gen_mare, 2)},
    {"Rețea": "UNet mic (16, 32, 64)", "Parametri": f"{nr_parametri(model_mic):,}", "Timp antrenare (min)": round(timp_mic / 60, 1),
     "Pierdere finală": round(pierderi_mic[-1], 5), "Timp DDIM-50 (s)": round(timp_gen_mic, 2)}])
display(tabel_bonus)
tabel_bonus.to_csv(os.path.join(TAB_DIR, "lab4_bonus.csv"), index=False)

fig, axs = plt.subplots(1, 2, figsize=(13, 6.6))
for ax, x, titlu in [(axs[0], x_mare, "UNet (32, 64, 128)"), (axs[1], x_mic, "UNet mic (16, 32, 64)")]:
    ax.imshow(grila_numpy(x))
    ax.set_title(f"{titlu}, DDIM 50 pași")
    ax.axis("off")
plt.tight_layout()
salveaza_fig(fig, "lab4_bonus_retea_mica")
plt.show()

După rulare, completez evaluarea vizuală pe baza acelorași eșantioane inițiale:

| Rețea | Calitate (1–5) | Observații din imaginile mele |
|---|---:|---|
| U-Net (32, 64, 128) |  |  |
| U-Net mic (16, 32, 64) |  |  |

## 9. Întrebări de control — Laboratorul 4

**1. De ce procesul direct nu are parametri învățați și ce rol are?**  
Procesul direct este definit de programul `beta`, deci nu conține greutăți de rețea. Îl folosesc pentru a crea perechi de antrenare: imagine zgomotată și zgomotul cunoscut care i-a fost adăugat.

**2. De ce pot calcula direct imaginea la pasul `t`?**  
Zgomotele gaussiene succesive se pot combina într-un singur zgomot gaussian, iar produsele factorilor `alpha` sunt reprezentate prin `alpha_bar`. Formula rezultată dă imaginea la pasul dorit fără să calculez pașii precedenți.

**3. De ce primește U-Net-ul și pasul `t`?**  
Pasul indică nivelul de zgomot. Fără el, aceeași rețea ar trebui să ghicească dacă să curețe zgomot fin sau să reconstruiască structura din zgomot intens.

**4. De ce generarea este mai lentă decât un pas de antrenare?**  
Generarea DDPM face până la 1000 de predicții secvențiale; fiecare depinde de imaginea produsă la pasul anterior. Un pas de antrenare face o predicție pentru un singur pas ales aleator.

**5. Ce diferență văd între programele liniar și cosinus?**  
În procesul direct compar curbele de semnal păstrat și pasul pragului 0,5. După antrenare compar grilele produse cu același zgomot inițial. Îmi susțin concluzia prin aceste rezultate, deoarece pierderile brute nu au aceeași distribuție de niveluri de zgomot.

**6. Ce se schimbă când reduc pașii DDIM?**  
Reduc timpul de calcul, dar salturile dintre stări sunt mai mari și pot pierde detalii. Din rezultate aleg cel mai mic număr de pași la care imaginile rămân suficient de clare pentru scopul meu.

# PARTEA II — Laboratorul 5: Stable Diffusion în practică

În partea a doua folosesc Stable Diffusion 1.5, un model text-imagine preantrenat. Nu îl antrenez de la zero; schimb câte o setare și măsor efectul asupra imaginilor, scorului CLIP și timpului. Pentru această parte am nevoie de GPU, internet și spațiu pentru descărcarea modelului.

Înainte de rulare accept termenii modelului pe Hugging Face. Dacă accesul solicită autentificare, mă autentific în sesiunea Colab fără să scriu tokenul în notebook.

## 1. Încărcarea modelului și imaginile de bază

Încarc Stable Diffusion 1.5 în precizie `float16` pe GPU și verific metoda de eșantionare implicită. Generez setul de bază cu trei prompturi și trei seed-uri, la 30 de pași și ghidare 7,5. Aceste nouă imagini sunt reperul pentru experimentele A–E.

In [ ]:
for nume in ["model_lin", "model_cos", "model_mic", "MODELE"]:
    globals().pop(nume, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

import diffusers, transformers
from diffusers import (StableDiffusionPipeline, StableDiffusionImg2ImgPipeline, DDIMScheduler,
                       DPMSolverMultistepScheduler, EulerDiscreteScheduler)

assert torch.cuda.is_available(), "Laboratorul 5 necesită GPU (Runtime → Change runtime type → T4 GPU)"
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to("cuda")
pipe.set_progress_bar_config(disable=True)

SCHED_IMPLICIT = type(pipe.scheduler)        # metoda de eșantionare implicită a modelului
SCHED_CFG = pipe.scheduler.config            # configurația originală, folosită la schimbarea metodei

print("diffusers:", diffusers.__version__, "| transformers:", transformers.__version__)
print("Metoda de eșantionare implicită:", SCHED_IMPLICIT.__name__)
for nume, modul in [("Encoder de text (CLIP)", pipe.text_encoder), ("U-Net", pipe.unet), ("VAE", pipe.vae)]:
    print(f"{nume:24s} {sum(p.numel() for p in modul.parameters()) / 1e6:7.1f} M parametri")

### 1.1 Prompturile și configurația de bază

Folosesc aceleași trei prompturi în toate comparațiile. Pentru fiecare prompt generez imagini cu seed-urile 42, 7 și 123. Fixez rezoluția la 512×512, 30 de pași și `guidance_scale=7.5`.

In [ ]:
PROMPTURI = {
    "P1": "a watercolor painting of a red fox in a snowy forest",
    "P2": "a photo of a modern library interior, natural light",
    "P3": "a cartoon robot reading a book, flat illustration",
}
SEEDURI_BAZA = [42, 7, 123]
NEGATIV_TEXT = "blurry, low quality, text, watermark"
SCHEDULERI = {None: SCHED_IMPLICIT, "DDIM": DDIMScheduler,
              "DPM-Solver": DPMSolverMultistepScheduler, "Euler": EulerDiscreteScheduler}


def cfg(nume, pasi=30, ghidare=7.5, sched=None, negativ=None, prompturi=None, seeduri=None):
    # O configurație de generare; valorile implicite sunt cele ale configurației de bază
    return {"nume": nume, "pasi": pasi, "ghidare": ghidare, "sched": sched, "negativ": negativ,
            "prompturi": prompturi or PROMPTURI, "seeduri": seeduri or SEEDURI_BAZA}


def cale_img(nume_cfg, pid, seed):
    return os.path.join(IMG5_DIR, f"{nume_cfg}_{pid}_s{seed}.png")


_incalzit = False


def genereaza(c):
    # Generează (sau încarcă) imaginile unei configurații. Întoarce ({(prompt, seed): imagine}, informații)
    global _incalzit
    cale_info = os.path.join(IMG5_DIR, f"{c['nume']}_gen.json")
    chei = [(pid, s) for pid in c["prompturi"] for s in c["seeduri"]]
    if os.path.exists(cale_info) and not FORTEAZA and all(os.path.exists(cale_img(c["nume"], p, s)) for p, s in chei):
        return {(p, s): Image.open(cale_img(c["nume"], p, s)).convert("RGB") for p, s in chei}, incarca_json(cale_info)

    pipe.scheduler = SCHEDULERI[c["sched"]].from_config(SCHED_CFG)       # metoda de eșantionare a configurației
    if not _incalzit:                                                    # încălzire GPU: nu se măsoară
        pipe("warm-up", num_inference_steps=2, generator=torch.Generator("cuda").manual_seed(0))
        _incalzit = True
    imagini, timpi, filtru = {}, {}, {}
    for pid, s in chei:
        gen = torch.Generator("cuda").manual_seed(s)
        sincronizeaza()
        t0 = time.perf_counter()
        out = pipe(prompt=c["prompturi"][pid], negative_prompt=c["negativ"], num_inference_steps=c["pasi"],
                   guidance_scale=c["ghidare"], height=512, width=512, generator=gen)
        sincronizeaza()
        timpi[f"{pid}_s{s}"] = time.perf_counter() - t0
        filtru[f"{pid}_s{s}"] = bool(out.nsfw_content_detected[0]) if out.nsfw_content_detected else False
        out.images[0].save(cale_img(c["nume"], pid, s))
        imagini[(pid, s)] = out.images[0]
    info = {"timpi": timpi, "filtru": filtru}
    salveaza_json(info, cale_info)
    return imagini, info


def arata_grila(randuri, coloane, titlu, nume_fig, dim=3.0):
    # Grilă comparativă: rândurile sunt configurații (eticheta, nume), coloanele sunt perechi (prompt, seed)
    fig, axs = plt.subplots(len(randuri), len(coloane), figsize=(dim * len(coloane), dim * len(randuri) + 0.6), squeeze=False)
    for r, (eticheta, nume_cfg) in enumerate(randuri):
        cale_rez = os.path.join(IMG5_DIR, f"{nume_cfg}_rez.json")
        scoruri = incarca_json(cale_rez)["scoruri"] if os.path.exists(cale_rez) else {}
        for c, (pid, s) in enumerate(coloane):
            ax = axs[r, c]
            ax.imshow(Image.open(cale_img(nume_cfg, pid, s)))
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(f"{pid}, seed {s}", fontsize=10)
            if f"{pid}_s{s}" in scoruri:
                ax.set_xlabel(f"CLIP {scoruri[f'{pid}_s{s}']:.1f}", fontsize=9)
        axs[r, 0].set_ylabel(eticheta, fontsize=10)
    fig.suptitle(titlu, y=1.0, fontsize=13)
    plt.tight_layout()
    salveaza_fig(fig, nume_fig)
    plt.show()


CFG_BAZA = cfg("baza")
imgs_baza, info_baza = genereaza(CFG_BAZA)
print(f"{len(imgs_baza)} imagini de bază | timp mediu / imagine: {np.mean(list(info_baza['timpi'].values())):.2f} s")

fig, axs = plt.subplots(3, 3, figsize=(10, 10.6))
for r, pid in enumerate(PROMPTURI):
    for c, s in enumerate(SEEDURI_BAZA):
        axs[r, c].imshow(imgs_baza[(pid, s)])
        axs[r, c].set_xticks([]); axs[r, c].set_yticks([])
        if r == 0:
            axs[r, c].set_title(f"seed {s}")
    axs[r, 0].set_ylabel(pid, fontsize=12)
fig.suptitle("Configurația de bază: 30 pași, ghidare 7,5, 512×512", y=0.99, fontsize=13)
plt.tight_layout()
salveaza_fig(fig, "lab5_s1_grila_baza")
plt.show()

## 2. Instrumentele mele de măsurare

Folosesc scorul CLIP pentru potrivirea imaginii cu promptul și `time.perf_counter()` pentru timpul mediu pe imagine. Exclud generarea de încălzire din cronometrare și sincronizez GPU-ul înainte să citesc timpul.

Pentru fiecare configurație calculez media și abaterea standard CLIP pe cele nouă imagini. Interpretez scorul împreună cu grilele: CLIP nu măsoară fidel estetica, corectitudinea detaliilor sau toate aspectele de calitate. Păstrez separat și numărul imaginilor înlocuite de verificatorul de siguranță.

In [ ]:
from torchmetrics.functional.multimodal import clip_score

CLIP_MODEL = "openai/clip-vit-base-patch16"

# clip_score reîncarcă modelul CLIP la fiecare apel; memorarea lui în cache accelerează calculul
try:
    _mod = sys.modules["torchmetrics.functional.multimodal.clip_score"]
    _mod._get_clip_model_and_processor = functools.lru_cache(maxsize=2)(_mod._get_clip_model_and_processor)
except Exception:
    pass

_clip_alt = {}
_foloseste_alt = False


def _clip_direct(imagine, prompt):
    # Aceeași formulă (100 * cos(imagine, text), minim 0), calculată direct cu transformers
    from transformers import CLIPModel, CLIPProcessor
    if not _clip_alt:
        _clip_alt["m"] = CLIPModel.from_pretrained(CLIP_MODEL).to("cuda").eval()
        _clip_alt["p"] = CLIPProcessor.from_pretrained(CLIP_MODEL)
    m, p = _clip_alt["m"], _clip_alt["p"]
    with torch.no_grad():
        inp = p(text=[prompt], images=[imagine.convert("RGB")], return_tensors="pt", padding=True, truncation=True).to("cuda")
        img = m.visual_projection(m.vision_model(pixel_values=inp["pixel_values"]).pooler_output)
        txt = m.text_projection(m.text_model(input_ids=inp["input_ids"], attention_mask=inp["attention_mask"]).pooler_output)
        img = img / img.norm(dim=-1, keepdim=True)
        txt = txt / txt.norm(dim=-1, keepdim=True)
        return float(torch.clamp(100 * (img * txt).sum(-1), min=0).squeeze())


def scor_clip(imagine, prompt):
    # Scorul CLIP al unei imagini față de un text. Dacă clip_score nu merge cu versiunea instalată
    # de transformers, se trece automat la calculul direct (aceeași formulă).
    global _foloseste_alt
    if not _foloseste_alt:
        try:
            arr = torch.from_numpy(np.array(imagine.convert("RGB"))).permute(2, 0, 1)    # tensor uint8 (3, H, W)
            with torch.no_grad():
                return float(clip_score(arr.to("cuda"), prompt, model_name_or_path=CLIP_MODEL))
        except AttributeError:
            print("clip_score din torchmetrics nu este compatibil cu transformers; se folosește calculul direct.")
            _foloseste_alt = True
    return _clip_direct(imagine, prompt)


def ruleaza(c):
    # Rulează o configurație: imagini, scoruri CLIP, timpi. Rezultatul se salvează și se reutilizează.
    cale = os.path.join(IMG5_DIR, f"{c['nume']}_rez.json")
    if os.path.exists(cale) and not FORTEAZA:
        return incarca_json(cale)
    imagini, info = genereaza(c)
    scoruri = {f"{pid}_s{s}": scor_clip(img, c["prompturi"][pid]) for (pid, s), img in imagini.items()}
    v = np.array(list(scoruri.values()))
    negre = [k for k, flag in info["filtru"].items() if flag]            # imagini înlocuite de filtru
    rez = {"nume": c["nume"], "pasi": c["pasi"], "ghidare": c["ghidare"],
           "sched": c["sched"] or SCHED_IMPLICIT.__name__, "negativ": c["negativ"], "n_imagini": len(v),
           "clip_mean": float(v.mean()), "clip_std": float(v.std(ddof=1)) if len(v) > 1 else 0.0,
           "timp_mean": float(np.mean(list(info["timpi"].values()))),
           "scoruri": scoruri, "negre": negre,
           "clip_mean_fara_negre": float(np.mean([x for k, x in scoruri.items() if k not in negre])) if len(negre) < len(v) else None}
    salveaza_json(rez, cale)
    return rez


def afiseaza(eticheta, r):
    print(f"{eticheta:28s} CLIP {r['clip_mean']:.2f} ± {r['clip_std']:.2f} | {r['timp_mean']:.2f} s / imagine"
          + (f" | imagini înlocuite de filtru: {len(r['negre'])}" if r["negre"] else ""))


REZ = {"baza": ruleaza(CFG_BAZA)}                    # toate rezultatele, după numele configurației
afiseaza("Bază (30 pași, ghidare 7,5)", REZ["baza"])
print("Scorul CLIP pe imagine:", {k: round(v, 1) for k, v in REZ["baza"]["scoruri"].items()})

## 3. Experimente controlate A–E

În fiecare experiment schimb un singur parametru față de configurația de bază. Păstrez prompturile și seed-urile, cu excepția experimentului D, care testează seed-ul. Rezultatele sunt memorate pe disc pentru a evita repetarea generărilor la rerularea notebookului.

In [ ]:
COL_STANDARD = [("P1", 42), ("P2", 42), ("P3", 42)]       # coloanele grilelor comparative

### 3.1 Experimentul A — scara de ghidare

Testez valorile 1, 3, 7,5, 12 și 20 pentru `guidance_scale`. Compar cât de bine urmează imaginile promptul și urmăresc posibile schimbări de saturație, contrast și artefacte.

In [ ]:
VALORI_A = [1, 3, 7.5, 12, 20]
CFG_A = {v: (CFG_BAZA if v == 7.5 else cfg(f"A_ghidare_{v}", ghidare=v)) for v in VALORI_A}
for v, c in CFG_A.items():
    REZ[c["nume"]] = ruleaza(c)
    afiseaza(f"ghidare {v}", REZ[c["nume"]])
arata_grila([(f"ghidare {v}", c["nume"]) for v, c in CFG_A.items()], COL_STANDARD,
            "Experiment A: scara de ghidare (30 pași)", "lab5_A_grila")

### 3.2 Experimentul B — numărul de pași

Testez 5, 10, 20 și 50 de pași. Mă uit la compromisul dintre timpul de generare și claritatea imaginilor, folosind configurația de bază cu 30 de pași ca reper.

In [ ]:
VALORI_B = [5, 10, 20, 50]
CFG_B = {n: cfg(f"B_pasi_{n}", pasi=n) for n in VALORI_B}
for n, c in CFG_B.items():
    REZ[c["nume"]] = ruleaza(c)
    afiseaza(f"{n} pași", REZ[c["nume"]])
randuri_B = [(f"{n} pași", CFG_B[n]["nume"]) for n in (5, 10, 20)] + [("30 pași (bază)", "baza"), ("50 pași", CFG_B[50]["nume"])]
arata_grila(randuri_B, COL_STANDARD, "Experiment B: numărul de pași (ghidare 7,5)", "lab5_B_grila")

### 3.3 Experimentul C — metoda de eșantionare

Compar DDIM, DPM-Solver multistep și Euler la 10 și 30 de pași. Păstrez setările de bază și schimb doar schedulerul și numărul de pași indicat.

In [ ]:
METODE_C = ["DDIM", "DPM-Solver", "Euler"]
PASI_C = [10, 30]
CFG_C = {(m, n): cfg(f"C_{m}_{n}", pasi=n, sched=m) for m in METODE_C for n in PASI_C}
for (m, n), c in CFG_C.items():
    REZ[c["nume"]] = ruleaza(c)
    afiseaza(f"{m}, {n} pași", REZ[c["nume"]])
randuri_C = [(f"{m}, {n} pași", CFG_C[(m, n)]["nume"]) for n in PASI_C for m in METODE_C]
randuri_C.append((f"{SCHED_IMPLICIT.__name__.replace('Scheduler', '')} (implicit), 30 pași", "baza"))
arata_grila(randuri_C, COL_STANDARD, "Experiment C: metoda de eșantionare", "lab5_C_grila", dim=2.6)

### 3.4 Experimentul D — seed-ul

Generez promptul P1 cu seed-urile 42, 7, 123 și 999. Apoi generez de două ori cu seed 42 și compar pixelii pentru a verifica reproductibilitatea. Compar și două seed-uri diferite.

In [ ]:
P1_DOAR = {"P1": PROMPTURI["P1"]}
SEEDURI_D = [42, 7, 123, 999]
CFG_D = cfg("D_seeduri", prompturi=P1_DOAR, seeduri=SEEDURI_D)
CFG_D1 = cfg("D_repetare_1", prompturi=P1_DOAR, seeduri=[42])
CFG_D2 = cfg("D_repetare_2", prompturi=P1_DOAR, seeduri=[42])
for c in (CFG_D, CFG_D1, CFG_D2):
    REZ[c["nume"]] = ruleaza(c)
afiseaza("P1, 4 seed-uri", REZ["D_seeduri"])
arata_grila([("P1, 4 seed-uri", "D_seeduri")], [("P1", s) for s in SEEDURI_D],
            "Experiment D: același prompt (P1), seed-uri diferite", "lab5_D_grila", dim=3.4)


def pixeli(nume_cfg, pid, seed):
    return np.array(Image.open(cale_img(nume_cfg, pid, seed)).convert("RGB")).astype(np.float32)


def dif_medie(a, b):
    return float(np.abs(a - b).mean())


verificare_seed = {
    "același seed, două generări (42 și 42)": dif_medie(pixeli("D_repetare_1", "P1", 42), pixeli("D_repetare_2", "P1", 42)),
    "seed-uri diferite (42 și 7)": dif_medie(pixeli("D_seeduri", "P1", 42), pixeli("D_seeduri", "P1", 7)),
}
for k, v in verificare_seed.items():
    print(f"Diferență medie între pixeli, {k}: {v:.4f}")
salveaza_json(verificare_seed, os.path.join(TAB_DIR, "lab5_verificare_seed.json"))
print("Scor CLIP P1 pe seed-uri:", {k: round(v, 2) for k, v in REZ["D_seeduri"]["scoruri"].items()})

### 3.5 Experimentul E — promptul negativ

Compar configurația de bază cu aceeași configurație, dar cu promptul negativ `blurry, low quality, text, watermark`. Verific dacă se schimbă scorul CLIP și dacă observ diferențe vizuale.

In [ ]:
CFG_E = cfg("E_negativ", negativ=NEGATIV_TEXT)
REZ["E_negativ"] = ruleaza(CFG_E)
afiseaza("fără prompt negativ", REZ["baza"])
afiseaza("cu prompt negativ", REZ["E_negativ"])
arata_grila([("fără prompt negativ", "baza"), ("cu prompt negativ", "E_negativ")], COL_STANDARD,
            "Experiment E: promptul negativ", "lab5_E_grila")

### 3.6 Tabelul de rezultate

Tabelul reunește scorul mediu CLIP, abaterea standard, timpul, diferența față de configurația de bază și numărul imaginilor înlocuite. Compar și scorurile pe perechi (același prompt și seed) ca să reduc variația introdusă de conținutul prompturilor.

O diferență mai mică decât variația observată nu este suficientă pentru a susține o concluzie fermă. Formulez recomandarea în funcție de scor, timp și inspecția imaginilor.

In [ ]:
RANDURI = [("Bază", "30 pași, ghidare 7,5", "baza")]
RANDURI += [("A", f"ghidare {v}", c["nume"]) for v, c in CFG_A.items() if v != 7.5]
RANDURI += [("B", f"{n} pași", CFG_B[n]["nume"]) for n in VALORI_B]
RANDURI += [("C", f"{m}, {n} pași", CFG_C[(m, n)]["nume"]) for n in PASI_C for m in METODE_C]
RANDURI += [("E", "prompt negativ", "E_negativ")]


def rand_tabel(exp, valoare, nume_cfg):
    r, b = REZ[nume_cfg], REZ["baza"]
    linie = {"Experiment": exp, "Valoare": valoare, "Scor CLIP (medie ± abatere)": f"{r['clip_mean']:.2f} ± {r['clip_std']:.2f}",
             "Timp / imagine (s)": round(r["timp_mean"], 2), "Δ față de bază": "—", "Concludent (σ)": "—",
             "Δ pe perechi": "—", "Concludent (perechi)": "—", "Imagini înlocuite": len(r["negre"])}
    if nume_cfg != "baza":
        delta = r["clip_mean"] - b["clip_mean"]
        d = np.array([r["scoruri"][k] - b["scoruri"][k] for k in r["scoruri"]])       # diferențe pe perechi
        eroare = d.std(ddof=1) / np.sqrt(len(d))
        linie.update({"Δ față de bază": f"{delta:+.2f}",
                      "Concludent (σ)": "Da" if abs(delta) > max(r["clip_std"], b["clip_std"]) else "Nu",
                      "Δ pe perechi": f"{d.mean():+.2f} ± {eroare:.2f}",
                      "Concludent (perechi)": "Da" if abs(d.mean()) > 2 * eroare else "Nu"})
    return linie


linii = [rand_tabel(*r) for r in RANDURI]
rD = REZ["D_seeduri"]
linii.append({"Experiment": "D", "Valoare": "4 seed-uri, P1 (42, 7, 123, 999)",
              "Scor CLIP (medie ± abatere)": f"{rD['clip_mean']:.2f} ± {rD['clip_std']:.2f}",
              "Timp / imagine (s)": round(rD["timp_mean"], 2), "Δ față de bază": "—", "Concludent (σ)": "—",
              "Δ pe perechi": "—", "Concludent (perechi)": "—", "Imagini înlocuite": len(rD["negre"])})
d_rep = verificare_seed["același seed, două generări (42 și 42)"]
linii.append({"Experiment": "D", "Valoare": f"seed 42 de două ori (diferență pixeli = {d_rep:.4f})",
              "Scor CLIP (medie ± abatere)": f"{REZ['D_repetare_1']['clip_mean']:.2f} și {REZ['D_repetare_2']['clip_mean']:.2f}",
              "Timp / imagine (s)": round((REZ["D_repetare_1"]["timp_mean"] + REZ["D_repetare_2"]["timp_mean"]) / 2, 2),
              "Δ față de bază": "—", "Concludent (σ)": "—", "Δ pe perechi": "—", "Concludent (perechi)": "—",
              "Imagini înlocuite": len(REZ["D_repetare_1"]["negre"]) + len(REZ["D_repetare_2"]["negre"])})

tabel5 = pd.DataFrame(linii)
pd.set_option("display.width", 250)
display(tabel5)
tabel5.to_csv(os.path.join(TAB_DIR, "lab5_tabel_rezultate.csv"), index=False)

alt = tabel5[tabel5["Experiment"].isin(["A", "B", "C", "E"])]
print("\nDiferențe concludente după regula σ:",
      ", ".join(f"{r['Experiment']} {r['Valoare']} ({r['Δ față de bază']})" for _, r in alt[alt["Concludent (σ)"] == "Da"].iterrows()) or "niciuna")
print("Diferențe concludente pe perechi:",
      ", ".join(f"{r['Experiment']} {r['Valoare']} ({r['Δ pe perechi']})" for _, r in alt[alt["Concludent (perechi)"] == "Da"].iterrows()) or "niciuna")
print("Celelalte diferențe se încadrează în variația dintre imagini și nu se interpretează ca efect real.")

După rulare, îmi notez observațiile vizuale aici. Le compar cu CLIP și cu timpul din tabel; nu deduc calitatea doar dintr-un scor.

| Experiment | Valoare | Observațiile mele |
|---|---|---|
| Bază | 30 pași, ghidare 7,5 |  |
| A | ghidare 1 / 3 / 12 / 20 |  |
| B | 5 / 10 / 20 / 50 pași |  |
| C | DDIM / DPM-Solver / Euler |  |
| D | seed-uri diferite / repetat |  |
| E | prompt negativ |  |

### 3.7 Graficele experimentelor A și B

Trasez scorul CLIP mediu cu bare de abatere standard pentru ghidare și numărul de pași. Graficul de pași include și timpul pe imagine pentru comparația de viteză.

In [ ]:
def grafic_param(valori, nume_cfg, xlabel, titlu, nume_fig):
    medii = [REZ[n]["clip_mean"] for n in nume_cfg]
    abateri = [REZ[n]["clip_std"] for n in nume_cfg]
    fig, ax = plt.subplots(figsize=(7, 4.3))
    ax.errorbar(valori, medii, yerr=abateri, fmt="-o", capsize=5, color="tab:blue", ecolor="gray")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("scor CLIP mediu (± abatere standard)")
    ax.set_title(titlu)
    ax.set_xticks(valori)
    ax.grid(alpha=0.3)
    salveaza_fig(fig, nume_fig)
    plt.show()


grafic_param(VALORI_A, [CFG_A[v]["nume"] for v in VALORI_A], "scara de ghidare",
             "Experiment A: scorul CLIP în funcție de scara de ghidare", "lab5_A_grafic")

pasi_B = sorted(VALORI_B + [30])                                   # se adaugă și configurația de bază (30 pași)
nume_B = ["baza" if n == 30 else CFG_B[n]["nume"] for n in pasi_B]
grafic_param(pasi_B, nume_B, "număr de pași", "Experiment B: scorul CLIP în funcție de numărul de pași", "lab5_B_grafic")

fig, ax = plt.subplots(figsize=(7, 4.3))
ax.plot(pasi_B, [REZ[n]["timp_mean"] for n in nume_B], "-o", color="tab:red")
ax.axhline(2.0, color="gray", ls="--", lw=1, label="limită 2 s")
ax.set_xlabel("număr de pași")
ax.set_ylabel("timp mediu pe imagine (s)")
ax.set_title("Timpul de generare în funcție de numărul de pași")
ax.set_xticks(pasi_B)
ax.grid(alpha=0.3)
ax.legend()
salveaza_fig(fig, "lab5_B_timp")
plt.show()

### 3.8 Verificatorul de siguranță

Afișez configurațiile pentru care verificatorul a înlocuit imagini și compar scorul CLIP cu și fără acestea. Astfel văd cum afectează filtrarea agregatele.

In [ ]:
linii = [{"Configurație": nume, "Imagini înlocuite": ", ".join(r["negre"]), "CLIP (toate)": round(r["clip_mean"], 2),
          "CLIP (fără ele)": round(r["clip_mean_fara_negre"], 2) if r["clip_mean_fara_negre"] is not None else None}
         for nume, r in REZ.items() if r["negre"]]
if linii:
    tabel_negre = pd.DataFrame(linii)
    display(tabel_negre)
    tabel_negre.to_csv(os.path.join(TAB_DIR, "lab5_imagini_negre.csv"), index=False)
else:
    print("Verificatorul de siguranță nu a înlocuit nicio imagine.")

## 4. Recomandarea mea

Pe baza tabelului, a graficelor și a imaginilor, formulez două recomandări: configurația pe care o consider cea mai bună vizual și configurația care se încadrează în limita de timp de 2 secunde pe imagine. Completez recomandările după rularea experimentelor și precizez orice compromis sau limitare a măsurătorilor.

In [ ]:
CFG_F = {n: cfg(f"F_DPM-Solver_{n}", pasi=n, sched="DPM-Solver") for n in (6, 8)}
REZ_F = {}
for n, c in CFG_F.items():
    REZ_F[c["nume"]] = ruleaza(c)
    afiseaza(f"DPM-Solver, {n} pași", REZ_F[c["nume"]])
arata_grila([(f"DPM-Solver, {n} pași", c["nume"]) for n, c in CFG_F.items()] +
            [("DPM-Solver, 10 pași", CFG_C[("DPM-Solver", 10)]["nume"])],
            COL_STANDARD, "DPM-Solver la 6, 8 și 10 pași", "lab5_F_grila")

### 4.1 Verificarea configurațiilor rapide

Compar și DPM-Solver la 6 și 8 pași pentru a verifica dacă obțin o marjă mai mare sub limita de 2 secunde. Aceste configurații suplimentare sunt evaluate cu aceleași prompturi și seed-uri.

In [ ]:
def eticheta(r):
    met = r["sched"].replace("MultistepScheduler", "").replace("DiscreteScheduler", "").replace("Scheduler", "")
    return f"{met}, {r['pasi']} pași, ghidare {r['ghidare']}" + (", prompt neg." if r["negativ"] else "")


TOATE = {**REZ, **REZ_F}
clasament = pd.DataFrame([{"Configurație": eticheta(r), "CLIP": r["clip_mean"], "σ": r["clip_std"],
                           "Timp (s)": r["timp_mean"], "Imagini înlocuite": len(r["negre"])}
                          for r in TOATE.values() if r["n_imagini"] == 9]).drop_duplicates("Configurație")
clasament["Scor CLIP (medie ± abatere)"] = clasament.apply(lambda r: f"{r['CLIP']:.2f} ± {r['σ']:.2f}", axis=1)
clasament["Timp / imagine (s)"] = clasament["Timp (s)"].round(2)
COL = ["Configurație", "Scor CLIP (medie ± abatere)", "Timp / imagine (s)", "Imagini înlocuite"]

print("Primele 5 configurații după scorul CLIP:")
display(clasament.sort_values("CLIP", ascending=False)[COL].head(5).reset_index(drop=True))
print("Configurații cu timp sub 2 s, după scorul CLIP:")
display(clasament[clasament["Timp (s)"] < 2.0].sort_values("CLIP", ascending=False)[COL].head(8).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.errorbar(clasament["Timp (s)"], clasament["CLIP"], yerr=clasament["σ"], fmt="o", color="tab:blue", ecolor="lightgray", capsize=3)
for _, r in clasament.iterrows():
    ax.annotate(r["Configurație"].replace(", ghidare 7.5", "").replace(" pași", "p"), (r["Timp (s)"], r["CLIP"]),
                fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.axvline(2.0, color="red", ls="--", lw=1, label="limită 2 s")
ax.set_xlabel("timp mediu pe imagine (s)")
ax.set_ylabel("scor CLIP mediu (± abatere standard)")
ax.set_title("Compromisul dintre calitate și viteză")
ax.grid(alpha=0.3)
ax.legend()
salveaza_fig(fig, "lab5_s4_calitate_viteza")
plt.show()
clasament[["Configurație", "CLIP", "σ", "Timp (s)", "Imagini înlocuite"]].to_csv(os.path.join(TAB_DIR, "lab5_clasament.csv"), index=False)

### Rezumatul recomandărilor după rulare

**Pentru calitate vizuală:** completez metoda, numărul de pași și ghidarea, apoi justific alegerea prin grile, scor CLIP și timp.

**Pentru viteză sub 2 secunde/imagine:** completez configurația care respectă limita în măsurarea mea și precizez marja față de 2 secunde. Menționez dacă rezultatul rămâne coerent vizual.

Nu copiez cifre din exemplu; valorile trebuie preluate din tabelele generate de rularea mea.

## 5. Opțional: img2img

Pornesc de la imaginea P1 generată în configurația de bază și testez `strength` 0,3, 0,6 și 0,9. Parametrul controlează cât zgomot se adaugă imaginii de intrare înainte de procesul invers: la valori mici păstrez mai mult din imagine, iar la valori mari permit transformări mai ample.

Legătura cu Laboratorul 4 este procesul direct, aplicat aici în spațiul latent. Compar imaginile și calculez MSE față de imaginea inițială.

In [ ]:
STRENGTHS = [0.3, 0.6, 0.9]
cale_i2i = os.path.join(IMG5_DIR, "img2img_rez.json")
init_img = imgs_baza[("P1", 42)]
cale_poze = [os.path.join(IMG5_DIR, f"img2img_strength_{s}.png") for s in STRENGTHS]

if os.path.exists(cale_i2i) and not FORTEAZA and all(os.path.exists(c) for c in cale_poze):
    rez_i2i = incarca_json(cale_i2i)
else:
    pipe.scheduler = SCHED_IMPLICIT.from_config(SCHED_CFG)
    pipe_i2i = StableDiffusionImg2ImgPipeline.from_pipe(pipe)          # refolosește componentele deja încărcate
    pipe_i2i.set_progress_bar_config(disable=True)
    rez_i2i = {}
    for s, cale_poza in zip(STRENGTHS, cale_poze):
        sincronizeaza()
        t0 = time.perf_counter()
        img = pipe_i2i(prompt=PROMPTURI["P1"], image=init_img, strength=s, num_inference_steps=30, guidance_scale=7.5,
                       generator=torch.Generator("cuda").manual_seed(42)).images[0]
        sincronizeaza()
        img.save(cale_poza)
        mse = float(np.mean((np.asarray(img, dtype=np.float32) / 255 - np.asarray(init_img, dtype=np.float32) / 255) ** 2))
        rez_i2i[str(s)] = {"clip": scor_clip(img, PROMPTURI["P1"]), "timp": time.perf_counter() - t0,
                           "pasi": int(30 * s), "mse": mse}
    salveaza_json(rez_i2i, cale_i2i)
    del pipe_i2i

fig, axs = plt.subplots(1, 4, figsize=(15, 4.4))
axs[0].imshow(init_img)
axs[0].set_title("imaginea inițială")
for ax, s, cale_poza in zip(axs[1:], STRENGTHS, cale_poze):
    ax.imshow(Image.open(cale_poza))
    ax.set_title(f"strength {s} ({rez_i2i[str(s)]['pasi']} pași)")
    ax.set_xlabel(f"CLIP {rez_i2i[str(s)]['clip']:.1f} | MSE {rez_i2i[str(s)]['mse']:.3f}")
for ax in axs:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
salveaza_fig(fig, "lab5_img2img")
plt.show()

tabel_i2i = pd.DataFrame([{"strength": s, "Pași efectivi": rez_i2i[str(s)]["pasi"], "Scor CLIP": round(rez_i2i[str(s)]["clip"], 2),
                           "MSE față de imaginea inițială": round(rez_i2i[str(s)]["mse"], 4)}
                          for s in STRENGTHS])
display(tabel_i2i)
tabel_i2i.to_csv(os.path.join(TAB_DIR, "lab5_img2img.csv"), index=False)

## 6. Întrebări de control — Laboratorul 5

**1. Ce efect au ghidarea 1 și ghidarea 20?**  
La ghidare mică, textul influențează mai puțin generarea, iar la ghidare foarte mare modelul este împins puternic spre prompt. Verific în grile dacă apar imagini generice, saturație exagerată sau artefacte.

**2. De la câți pași nu mai observ îmbunătățiri clare?**  
Răspund comparând imaginile la 5, 10, 20, 30 și 50 de pași, apoi verific dacă scorul CLIP susține observația. Notez și situațiile în care CLIP nu reflectă bine calitatea vizuală.

**3. De ce pot fi DPM-Solver și Euler mai potrivite la puțini pași decât DDPM?**  
Aceste schedulere urmăresc soluții numerice ale procesului invers folosind pași mai eficienți. DDPM clasic este stocastic și este gândit pentru un șir lung de actualizări; cu foarte puțini pași poate pierde precizie.

**4. De ce același seed reproduce imaginea?**  
Seed-ul fixează zgomotul inițial. Cu același prompt, model, scheduler și setări, repet procesul pornind din același zgomot. Celula experimentului D verifică numeric acest lucru.

**5. Ce nu măsoară CLIP?**  
CLIP estimează potrivirea imagine-text, dar nu verifică complet anatomia, aspectul estetic, corectitudinea fiecărui detaliu sau eventualele defecte. De aceea îl compar cu inspecția vizuală.

**6. Ce riscuri etice sau juridice iau în calcul?**  
Verific licența modelului și drepturile asupra materialelor folosite. Evit generarea de persoane reale fără acord, dezinformare și conținut dăunător; țin cont de biasurile modelului și de faptul că scorul automat nu garantează utilizare responsabilă.

# PARTEA III — Concluziile mele

## Laboratorul 4

Am construit procesul direct și U-Net-ul, am antrenat două modele cu programe de zgomot diferite și am comparat generarea DDPM cu DDIM. Completez această concluzie după execuție, pe baza pierderilor, timpilor și grilelor:

- programul de zgomot pe care îl prefer și dovezile din rezultate: **de completat după rulare**;
- numărul de pași DDIM care oferă compromisul potrivit pentru mine: **de completat după rulare**;
- diferența pe care am observat-o între U-Net-ul de bază și cel mic: **de completat după rulare**.

## Laboratorul 5

Am testat efectul ghidării, numărului de pași, schedulerului, seed-ului și promptului negativ. Recomandările mele se bazează pe măsurătorile și imaginile efectiv generate:

- configurația pentru calitate: **de completat după rulare**;
- configurația pentru viteză: **de completat după rulare**;
- o limitare a scorului CLIP observată în experimente: **de completat după rulare**.

## Configurația tehnică

| Parte | Model și date | Antrenare / generare | Evaluare |
|---|---|---|---|
| Laboratorul 4 | DDPM cu U-Net; Fashion-MNIST grayscale 32×32 | Adam, rata 0,0002, lot 128, 15 epoci; DDPM și DDIM | MSE, grile, timp și statistici pixel |
| Laboratorul 5 | Stable Diffusion 1.5, condiționat de text, 512×512 | model preantrenat; experimente A–E și img2img opțional | CLIP, abatere standard, timp și inspecție vizuală |

## Funcția de pierdere

În Laboratorul 4 optimizez MSE dintre zgomotul real și cel prezis:

`L = mean(||epsilon - epsilon_theta(x_t, t)||^2)`

Aceasta oferă ținta de antrenare pentru fiecare imagine zgomotată. În Laboratorul 5 nu antrenez Stable Diffusion; folosesc CLIP ca măsură separată a potrivirii imaginii generate cu textul.

## Salvarea figurilor și tabelelor

La final, celulele de mai jos adună graficele și tabelele într-o arhivă. După rularea completă verific arhiva și notebookul înainte să predau.

In [ ]:
import zipfile

arhiva = "figuri_si_tabele_Lab4_5.zip"
with zipfile.ZipFile(arhiva, "w", zipfile.ZIP_DEFLATED) as z:
    for director in (FIG_DIR, TAB_DIR):
        for fisier in sorted(os.listdir(director)):
            z.write(os.path.join(director, fisier), os.path.join(os.path.basename(director), fisier))
print("Arhivă creată:", arhiva, f"({os.path.getsize(arhiva) / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(arhiva)
except ImportError:
    pass